# Notebook 02 — Biomechanics Analysis

Deep-dive into the 18 biomechanical features across the 4 running form classes.

**What we cover:**
1. Feature distributions by class (boxplots + KDE)
2. Trunk lean vs overstride scatter (key class separators)
3. Arm crossing analysis
4. Temporal profiles — how features change through the gait cycle
5. Feature correlation matrix
6. Statistical significance tests between classes

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

CLASS_PALETTE = {
    'good_form':    '#2ecc71',
    'overstriding': '#e74c3c',
    'forward_lean': '#f39c12',
    'arm_crossing': '#9b59b6',
}
CLASS_ORDER = ['good_form', 'overstriding', 'forward_lean', 'arm_crossing']

df = pd.read_csv('../data/processed/features/biomech_features.csv')
# Per-clip means for scatter plots
clip_means = df.groupby(['video_stem', 'form_class']).mean(numeric_only=True).reset_index()
print(f'Loaded {len(df):,} rows | {df["video_stem"].nunique()} clips')
print('Class counts:', df.groupby('form_class')['video_stem'].nunique().to_dict())

## 1. Key Feature Distributions by Class

In [ ]:
features_to_plot = [
    ('trunk_lean_angle',   'Trunk Lean (°)',         (3, 12)),
    ('max_overstride',     'Overstride (torso units)',(-0.15, 0.05)),
    ('arm_swing_symmetry', 'Arm Asymmetry (°)',       (0, 15)),
    ('hip_drop_angle',     'Hip Drop (°)',            (0, 5)),
    ('knee_drive_angle',   'Knee Drive (°)',          (60, 999)),
    ('vertical_oscillation','Vert. Oscillation',     (0, 0.08)),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (col, label, (lo, hi)) in zip(axes.flatten(), features_to_plot):
    if col not in df.columns:
        ax.set_visible(False); continue
    sns.boxplot(data=df[[col,'form_class']].dropna(),
                x='form_class', y=col, order=CLASS_ORDER,
                palette=CLASS_PALETTE, ax=ax, width=0.5)
    # Ideal range shading
    if hi < 999:
        ax.axhspan(lo, hi, alpha=0.08, color='green', label='Ideal range')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)

axes[0][0].legend(loc='upper right', fontsize=8)
plt.suptitle('Biomechanical Features by Running Form Class\n(green band = ideal range)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Key Class Separators — Scatter Plot

These two features together best distinguish the 4 form classes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trunk lean vs Overstride (separates overstriding + forward_lean from good)
for cls in CLASS_ORDER:
    sub = clip_means[clip_means['form_class'] == cls]
    axes[0].scatter(sub['max_overstride'], sub['trunk_lean_angle'],
                    color=CLASS_PALETTE[cls], label=cls, alpha=0.75, s=60, edgecolors='white')
axes[0].axvline(0.05, color='gray', linestyle='--', alpha=0.5, label='Overstride threshold')
axes[0].axhline(12,   color='gray', linestyle=':',  alpha=0.5, label='Trunk lean threshold')
axes[0].set_xlabel('Max Overstride (torso units)'); axes[0].set_ylabel('Trunk Lean (°)')
axes[0].set_title('Overstride vs Trunk Lean'); axes[0].legend(fontsize=8)

# Arm crossing vs Hip drop (separates arm_crossing from others)
for cls in CLASS_ORDER:
    sub = clip_means[clip_means['form_class'] == cls]
    axes[1].scatter(sub.get('left_arm_cross', 0), sub.get('hip_drop_angle', 0),
                    color=CLASS_PALETTE[cls], label=cls, alpha=0.75, s=60, edgecolors='white')
axes[1].axvline(0.05, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Left Arm Cross (torso units)'); axes[1].set_ylabel('Hip Drop (°)')
axes[1].set_title('Arm Crossing vs Hip Drop'); axes[1].legend(fontsize=8)

plt.suptitle('Feature Space — Class Separation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Temporal Profiles — How Features Change Over the Gait Cycle

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, form_class in zip(axes.flatten(), CLASS_ORDER):
    sub   = df[df['form_class'] == form_class]
    color = CLASS_PALETTE[form_class]
    
    for col, lbl, lc in [
        ('trunk_lean_angle',    'Trunk lean',    '#2980b9'),
        ('knee_drive_angle',    'Knee drive',    '#27ae60'),
        ('arm_swing_symmetry',  'Arm asymmetry', '#e67e22'),
    ]:
        if col not in sub.columns: continue
        profiles = []
        for _, grp in sub.groupby('video_stem'):
            vals = grp.sort_values('frame')[col].dropna().values
            if len(vals) > 5:
                profiles.append(np.interp(
                    np.linspace(0,1,50), np.linspace(0,1,len(vals)),
                    np.nan_to_num(vals, nan=float(np.nanmean(vals)))
                ))
        if profiles:
            arr  = np.array(profiles)
            mean = arr.mean(0); std = arr.std(0)
            x    = np.linspace(0, 1, 50)
            ax.plot(x, mean, color=lc, linewidth=2, label=lbl)
            ax.fill_between(x, mean-std, mean+std, color=lc, alpha=0.12)
    
    ax.set_title(form_class.replace('_',' ').title(), fontweight='bold', color=color)
    ax.set_xlabel('Gait cycle (normalized)')
    ax.set_ylabel('Angle (°)')
    ax.legend(fontsize=8)

plt.suptitle('Temporal Feature Profiles per Form Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Statistical Significance — Good Form vs Each Fault Class

In [ ]:
from scipy.stats import mannwhitneyu

key_features = ['trunk_lean_angle', 'max_overstride', 'arm_swing_symmetry',
                'hip_drop_angle', 'knee_drive_angle']
key_features = [f for f in key_features if f in clip_means.columns]

good = clip_means[clip_means['form_class'] == 'good_form']

print('Mann-Whitney U test: good_form vs each fault class')
print('(p < 0.05 = statistically significant difference)')
print('='*65)

for fault_class in ['overstriding', 'forward_lean', 'arm_crossing']:
    fault = clip_means[clip_means['form_class'] == fault_class]
    print(f'\nvs {fault_class}:')
    for feat in key_features:
        g_vals = good[feat].dropna()
        f_vals = fault[feat].dropna()
        if len(g_vals) < 3 or len(f_vals) < 3: continue
        _, p = mannwhitneyu(g_vals, f_vals, alternative='two-sided')
        sig  = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        g_m  = g_vals.mean(); f_m = f_vals.mean()
        print(f'  {feat:25s}: good={g_m:6.2f}, {fault_class[:10]}={f_m:6.2f}  p={p:.4f} {sig}')

## 5. Feature Correlation Matrix

In [ ]:
numeric_feats = [
    'trunk_lean_angle', 'max_overstride', 'arm_swing_symmetry',
    'hip_drop_angle', 'knee_drive_angle', 'stride_angle',
    'vertical_oscillation', 'left_arm_cross', 'right_arm_cross', 'head_alignment',
]
available = [f for f in numeric_feats if f in clip_means.columns]

corr = clip_means[available].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix (per-clip means)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Most correlated pairs
high = []
for i in range(len(corr)):
    for j in range(i+1, len(corr)):
        r = abs(corr.iloc[i,j])
        if r > 0.5:
            high.append((corr.columns[i], corr.columns[j], round(corr.iloc[i,j],3)))
if high:
    print('High correlation pairs (|r| > 0.5):')
    for a, b, r in sorted(high, key=lambda x: -abs(x[2])):
        print(f'  {a:30s} ↔ {b:30s}  r={r}')